# Modeling

**Approach:** Logistic regression and XGBoost on matchup feature differences; Elo-style rating model for a third probability stream. **Evaluation:** Log loss. **Validation:** Time-based CV by season (train on past seasons, validate on one season)—never use a random split.

In [48]:
import sys
sys.path.insert(0, '..')
import importlib
import src.features
import src.data_loading
import src.splits
import src.train_logreg
import src.train_xgb
import src.matchup_builder
import src.predict_2026
import src.named_matchups
import pandas as pd

importlib.reload(src.features)
importlib.reload(src.data_loading)
importlib.reload(src.splits)
importlib.reload(src.train_logreg)
importlib.reload(src.train_xgb)
importlib.reload(src.matchup_builder)
importlib.reload(src.predict_2026)
importlib.reload(src.named_matchups)


<module 'src.named_matchups' from '/Users/calebrh/model-madness/march-machine-learning-mania-2026/notebooks/../src/named_matchups.py'>

In [49]:
import sys
sys.path.insert(0, '..')
from src import validation, train_logreg, train_xgb

# Stubs: get_time_splits(seasons, n_splits), evaluate_cv(model, X, y, splits)
try:
    validation.get_time_splits([1985, 1986, 1987], n_splits=2)
except NotImplementedError:
    print("validation.get_time_splits: implement time-based folds.")

# Same time-based train/val split via `src.splits.split_matchup_train_val` (Season <= 2018 vs later).
train_logreg.train_logreg(None, None)
train_xgb.train_xgb()

validation.get_time_splits: implement time-based folds.
Baseline (6 feats) log loss: 0.5763305764083805
DROP  - efg_pct_diff           -> 0.5763711959019736
DROP  - tov_pct_diff           -> 0.5764323426168473
KEEP  + orb_pct_diff           -> 0.5761538002825893
KEEP  + ftr_diff               -> 0.5755634842761634
DROP  - last10_net_eff_diff    -> 0.5817279093161941
Final (8 feats) log loss: 0.5755634842761634
Final features: ['seed_diff', 'avg_margin_diff', 'off_eff_diff', 'def_eff_diff', 'net_eff_diff', 'win_pct_diff', 'orb_pct_diff', 'ftr_diff']
      target      pred  seed_diff  avg_margin_diff  off_eff_diff  \
2096       1  0.669373        0.0         8.512500     11.111882   
2097       0  0.330627        0.0        -8.512500    -11.111882   
2098       1  0.549169        0.0         0.136852      5.162797   
2099       0  0.450831        0.0        -0.136852     -5.162797   
2100       1  0.504676        0.0         2.022727      0.435941   
2101       0  0.495324        0.0    

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_parallel_tree=None, random_state=42, ...)

Testing building matchup data

In [50]:
from src.matchup_builder import build_matchup_data
from src.data_loading import load_team_names

matchups = build_matchup_data()
matchups.head()

print(matchups.isna().sum().sort_values(ascending=False))

team_names = load_team_names()
df_team_names = pd.DataFrame(team_names)
df_team_names.rename(columns={"TeamID": "TeamA", "TeamName": "TeamA_Name"}, inplace=True)
#df_team_names.rename(columns={"TeamID": "TeamB", "TeamName": "TeamB_Name"}, inplace=True)
#matchups = matchups.merge(df_team_names, on="TeamA", how="left")
df_team_names.rename(columns={"TeamA": "TeamB", "TeamA_Name": "TeamB_Name"}, inplace=True)
#matchups = matchups.merge(df_team_names, on="TeamB", how="left")

# print(matchups.head())
# print(matchups.shape)
# print(matchups["target"].value_counts())
# print(matchups.isna().sum().sort_values(ascending=False).head(10))

matchups = matchups.copy()
matchups["TeamLow"] = matchups[["TeamA", "TeamB"]].min(axis=1)
matchups["TeamHigh"] = matchups[["TeamA", "TeamB"]].max(axis=1)

# For each (Season, unordered team pair), show TeamA winner (target=1) before TeamA loser (target=0)
matchups = matchups.sort_values(
    ["Season", "DayNum", "TeamLow", "TeamHigh", "target", "TeamA", "TeamB"],
    ascending=[True, True, True, True, False, True, True],
).reset_index(drop=True)

matchups = matchups.drop(columns=["TeamLow", "TeamHigh"])
matchups.head(60)


Season                   0
efg_pct_diff             0
lastx_net_eff_diff       0
lastx_def_eff_diff       0
lastx_off_eff_diff       0
lastx_avg_margin_diff    0
lastx_win_pct_diff       0
ftr_diff                 0
orb_pct_diff             0
tov_pct_diff             0
net_eff_diff             0
DayNum                   0
def_eff_diff             0
off_eff_diff             0
avg_margin_diff          0
win_pct_diff             0
seed_diff                0
target                   0
TeamB                    0
TeamA                    0
last10_net_eff_diff      0
dtype: int64


,Season,DayNum,TeamA,TeamB,target,seed_diff,win_pct_diff,avg_margin_diff,off_eff_diff,def_eff_diff,...,efg_pct_diff,tov_pct_diff,orb_pct_diff,ftr_diff,lastx_win_pct_diff,lastx_avg_margin_diff,lastx_off_eff_diff,lastx_def_eff_diff,lastx_net_eff_diff,last10_net_eff_diff
0,2003,134,1421,1411,1,0.0,-0.151724,-9.208046,-1.053720,10.967252,...,-0.013235,0.014303,-0.012949,-0.152277,0.0,-0.3,6.427775,6.807102,-0.379327,-0.379327
1,2003,134,1411,1421,0,0.0,0.151724,9.208046,1.053720,-10.967252,...,0.013235,-0.014303,0.012949,0.152277,0.0,0.3,-6.427775,-6.807102,0.379327,0.379327
2,2003,136,1112,1436,1,-15.0,0.237685,10.309113,7.932821,-3.994784,...,0.022900,-0.022838,0.014010,0.031691,0.1,5.3,4.134648,-1.268476,5.403124,5.403124
3,2003,136,1436,1112,0,15.0,-0.237685,-10.309113,-7.932821,3.994784,...,-0.022900,0.022838,-0.014010,-0.031691,-0.1,-5.3,-4.134648,1.268476,-5.403124,-5.403124
4,2003,136,1113,1272,1,3.0,-0.172414,-1.896552,3.236790,5.700441,...,0.018997,0.005838,0.031277,0.071937,-0.3,-4.6,7.276878,13.624195,-6.347317,-6.347317
5,2003,136,1272,1113,0,-3.0,0.172414,1.896552,-3.236790,-5.700441,...,-0.018997,-0.005838,-0.031277,-0.071937,0.3,4.6,-7.276878,-13.624195,6.347317,6.347317
6,2003,136,1163,1140,1,-7.0,-0.041935,-0.140860,-1.407896,1.095328,...,-0.008439,0.007967,0.060507,-0.118209,-0.1,-0.1,-3.062419,-0.891669,-2.170749,-2.170749
7,2003,136,1140,1163,0,7.0,0.041935,0.140860,1.407896,-1.095328,...,0.008439,-0.007967,-0.060507,0.118209,0.1,0.1,3.062419,0.891669,2.170749,2.170749
8,2003,136,1141,1166,1,5.0,-0.085684,-8.805643,-4.400316,8.325189,...,0.005381,0.060548,0.019944,0.127689,0.1,4.7,11.920584,5.398172,6.522412,6.522412
9,2003,136,1166,1141,0,-5.0,0.085684,8.805643,4.400316,-8.325189,...,-0.005381,-0.060548,-0.019944,-0.127689,-0.1,-4.7,-11.920584,-5.398172,-6.522412,-6.522412


Saving matchup data

In [51]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # or Path("march-machine-learning-mania-2026")
output_dir = PROJECT_ROOT / "data" / "interim"

matchups.to_csv(output_dir / "men_train_matchups.csv", index=False)

Logistic Regression

In [52]:
from src.train_logreg import train_logreg
from src.train_xgb import train_xgb

model, logreg_feature_names = train_logreg()
xgb_model = train_xgb()

Baseline (6 feats) log loss: 0.5763305764083805
DROP  - efg_pct_diff           -> 0.5763711959019736
DROP  - tov_pct_diff           -> 0.5764323426168473
KEEP  + orb_pct_diff           -> 0.5761538002825893
KEEP  + ftr_diff               -> 0.5755634842761634
DROP  - last10_net_eff_diff    -> 0.5817279093161941
Final (8 feats) log loss: 0.5755634842761634
Final features: ['seed_diff', 'avg_margin_diff', 'off_eff_diff', 'def_eff_diff', 'net_eff_diff', 'win_pct_diff', 'orb_pct_diff', 'ftr_diff']
      target      pred  seed_diff  avg_margin_diff  off_eff_diff  \
2096       1  0.669373        0.0         8.512500     11.111882   
2097       0  0.330627        0.0        -8.512500    -11.111882   
2098       1  0.549169        0.0         0.136852      5.162797   
2099       0  0.450831        0.0        -0.136852     -5.162797   
2100       1  0.504676        0.0         2.022727      0.435941   
2101       0  0.495324        0.0        -2.022727     -0.435941   
2102       1  0.417377   

Save Model

In [53]:
import joblib
from src import config
from src.train_logreg import save_logreg_artifacts

save_logreg_artifacts(model, logreg_feature_names)
joblib.dump(xgb_model, config.MODELS_DIR / "xgb_men.pkl")

['/Users/calebrh/model-madness/march-machine-learning-mania-2026/models/xgb_men.pkl']

In [54]:
# Stage 2 (2026): all pairs from SampleSubmissionStage2 → logreg + XGB probabilities
from src.predict_2026 import predict_stage2_matchups

pred_df = predict_stage2_matchups()
pred_df.head(10)

,ID,pred_logreg,pred_xgb
0,2026_1101_1102,0.717459,0.742751
1,2026_1101_1103,0.262383,0.264509
2,2026_1101_1104,0.172273,0.122964
3,2026_1101_1105,0.485343,0.527932
4,2026_1101_1106,0.451871,0.452736
5,2026_1101_1107,0.399779,0.553334
6,2026_1101_1108,0.608113,0.600956
7,2026_1101_1110,0.359784,0.238072
8,2026_1101_1111,0.357443,0.161052
9,2026_1101_1112,0.068527,0.037467


In [55]:
# 2026 Round of 64: resolve names via MTeamSpellings → TeamID, then logreg + XGB
from src.named_matchups import predict_first_round_2026_from_default_list

first_round_df = predict_first_round_2026_from_default_list()
# Columns: matchup_line, team_a_name, team_b_name, team_a_id, team_b_id, ID, pred_logreg, pred_xgb
first_round_df[
    ["team_a_name", "team_b_name", "pred_logreg", "pred_xgb"]
]

first_round_df


,matchup_line,team_a_id,team_b_id,team_a_name,team_b_name,ID,pred_logreg,pred_xgb
0,(8) Ohio State vs. (9) TCU | 12:15 p.m. | CBS,1326,1395,Ohio St,TCU,2026_1326_1395,0.536654,0.446975
1,(4) Nebraska vs. (13) Troy | 12:40 p.m. | truTV,1304,1407,Nebraska,Troy,2026_1304_1407,0.852710,0.881629
2,(6) Louisville vs. (11) South Florida | 1:30 p...,1257,1378,Louisville,South Florida,2026_1257_1378,0.719733,0.648487
3,(5) Wisconsin vs. (12) High Point | 1:50 p.m. ...,1458,1219,Wisconsin,High Point,2026_1458_1219,0.623948,0.598413
4,(1) Duke vs. (16) Siena | 2:50 p.m. | CBS,1181,1373,Duke,Siena,2026_1181_1373,0.951789,0.970277
5,(5) Vanderbilt vs. (12) McNeese | 3:15 p.m. | ...,1435,1270,Vanderbilt,McNeese St,2026_1435_1270,0.717784,0.701331
6,(3) Michigan State vs. (14) North Dakota State...,1277,1295,Michigan St,N Dakota St,2026_1277_1295,0.830651,0.857205
7,(4) Arkansas vs. (13) Hawai'i | 4:25 p.m. | TBS,1116,1218,Arkansas,Hawaii,2026_1116_1218,0.781833,0.841665
8,(6) North Carolina vs. (11) VCU | 6:50 p.m. | TNT,1314,1433,North Carolina,VCU,2026_1314_1433,0.646455,0.732906
9,(1) Michigan vs. Howard | 7:10 p.m. | CBS,1276,1224,Michigan,Howard,2026_1276_1224,0.932770,0.966370


In [56]:
# 2026: ad-hoc bracket-style block (seed + team lines) → predictions
from src.named_matchups import predict_seed_block_matchups_2026

seed_block_text = """
1
Duke

9
TCU

5
St John's

4
Kansas

11
South Florida

3
Michigan St

7
UCLA

2
UConn

1
Florida

8
Clemson

5
Vanderbilt

4
Nebraska

Preview
TBD

11
VCU

3
Illinois

7
Saint Mary's

2
Houston

1
Arizona

9
Utah State

12
High Point

4
Arkansas

6
BYU

3
Gonzaga

7
Miami

2
Purdue

1
Michigan

9
Saint Louis

5
Texas Tech

13
Hofstra

6
Tennessee

3
Virginia

10
Santa Clara

2
Iowa State
""".strip()

bracket_df = predict_seed_block_matchups_2026(seed_block_text)
bracket_df[["team_a_name", "team_b_name", "pred_logreg", "pred_xgb"]]


,team_a_name,team_b_name,pred_logreg,pred_xgb
0,Duke,TCU,0.884635,0.843633
1,St John's,Kansas,0.520296,0.611971
2,South Florida,Michigan St,0.240610,0.151247
3,UCLA,Connecticut,0.239616,0.137065
4,Florida,Clemson,0.824702,0.780222
5,Vanderbilt,Nebraska,0.415989,0.453623
6,VCU,Illinois,0.145213,0.097166
7,St Mary's CA,Houston,0.269236,0.336062
8,Arizona,Utah St,0.808775,0.885084
9,High Point,Arkansas,0.324345,0.312198


XGBoost

In [57]:
# 2026: requested matchup set (keep prior seeds)
from src.named_matchups import predict_named_matchups_with_seeds

requested_matchups = """
(1) Duke vs. (5) St. John's
(1) Arizona vs. (4) Arkansas
(3) Gonzaga vs. (2) Purdue
(3) Michigan State vs. (2) UConn
(1) Florida vs. (4) Nebraska
(3) Illinois vs. (2) Houston
(6) Tennessee vs. (2) Iowa State
(1) Michigan vs. (5) Texas Tech
""".strip()

req_df = predict_named_matchups_with_seeds(requested_matchups)
req_df[["seed_a", "team_a_name", "seed_b", "team_b_name", "pred_logreg", "pred_xgb"]]


,seed_a,team_a_name,seed_b,team_b_name,pred_logreg,pred_xgb
0,1,Duke,5,St John's,0.781130,0.723589
1,1,Arizona,4,Arkansas,0.711102,0.792770
2,3,Gonzaga,2,Purdue,0.594270,0.581578
3,3,Michigan St,2,Connecticut,0.418932,0.333508
4,1,Florida,4,Nebraska,0.651925,0.531124
5,3,Illinois,2,Houston,0.458924,0.506501
6,6,Tennessee,2,Iowa St,0.277134,0.233269
7,1,Michigan,5,Texas Tech,0.730264,0.842998


In [ ]:
# Ad-hoc matchups (pred_* = P(team_a wins))
from src.named_matchups import predict_named_matchups

more_matchups = """
Nebraska vs. Houston
Michigan vs. Iowa State
Arizona vs. Gonzaga
Duke vs. Michigan St
""".strip()

more_df = predict_named_matchups(more_matchups)
more_df[["team_a_name", "team_b_name", "pred_logreg", "pred_xgb"]]
